# Credit Card Fraud Detection

XGBoost with SMOTE, threshold tuning, and feature importance

## 1. Install libraries

In [ ]:
!pip install xgboost imbalanced-learn -q

## 2. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

## 3. Create transaction data

In [ ]:
data = {
    "amount": [20, 50, 100, 10, 500, 30, 1000, 15, 70, 200, 25, 800, 40, 60, 900, 35, 120, 45, 700, 55],
    "time": [10, 12, 14, 15, 18, 20, 22, 25, 28, 30, 32, 35, 38, 40, 42, 45, 48, 50, 52, 55],
    "merchant_risk": [1, 1, 2, 1, 3, 1, 4, 1, 2, 2, 1, 4, 1, 2, 5, 1, 2, 1, 5, 2],
    "location_change": [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0],
    "previous_transactions": [20, 30, 25, 40, 15, 35, 10, 50, 30, 20, 45, 8, 50, 40, 5, 55, 30, 45, 4, 35],
    "fraud": [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0]
}

df = pd.DataFrame(data)
df

## 4. Separate input and output

In [ ]:
X = df.drop("fraud", axis=1)
y = df["fraud"]

## 5. Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 6. Apply SMOTE

In [ ]:
smote = SMOTE(random_state=42, k_neighbors=2)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(y_train.value_counts())

## 7. Create and train the XGBoost model

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

## 8. Get fraud probabilities

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
print(y_prob)

## 9. Evaluate using ROC-AUC

In [ ]:
auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", auc)

## 10. Tune the decision threshold

In [ ]:
threshold = 0.30
y_pred = (y_prob >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

A lower threshold can help detect more fraudulent transactions. The trade-off is that more normal transactions may be incorrectly marked as fraud.

## 11. Feature importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)
print(importance)

In [ ]:
importance.plot(kind="bar")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.title("XGBoost Feature Importance")
plt.show()

## Conclusion

XGBoost can be used to detect fraudulent transactions. SMOTE helps deal with the imbalanced training data by creating additional minority-class samples. Changing the decision threshold allows the model to focus more on detecting fraud. Feature importance scores show which transaction features were most useful to the model.

This small dataset is only for demonstration. A real fraud detection project should use a large dataset such as IEEE-CIS or another suitable transaction dataset and should evaluate the model with appropriate fraud detection metrics.